# 1. 문제 정의

서울시 따릉이 일별 수요 예측. [요구사항](../../docs/ml/REQUIREMENTS.md).

In [1]:
from pathlib import Path
import sys, json, time, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))
plt.rcParams.update({"figure.figsize": (10, 4), "font.family": "Malgun Gothic", "axes.unicode_minus": False})
OUT = ROOT / "outputs/ml"
OUT.mkdir(parents=True, exist_ok=True)
LOCAL = ROOT / "data/processed"
LOCAL.mkdir(parents=True, exist_ok=True)
paths = sorted((ROOT / "data/raw").glob("서울특별시 공공자전거 일별 대여건수_*.csv"))
daily = pd.concat([pd.read_csv(path, encoding="cp949") for path in paths], ignore_index=True)
daily = daily.rename(columns={"대여일자": "date", "대여건수": "rentals"})
daily["date"] = pd.to_datetime(daily.date)


In [2]:
contract = pd.DataFrame([
    ["예측 대상", "서울시 전체의 D일 총 대여건수 (회귀)"],
    ["예측 시점", "D-1일 00:00"],
    ["과거 정보", "D-2일까지의 확정 대여건수, D일 달력"],
    ["활용", "도시 전체 운영 규모 계획의 참고 지표"],
    ["데이터 범위", f"{daily.date.min():%Y-%m-%d}~{daily.date.max():%Y-%m-%d}, {len(daily)}일"],
    ["제약", "반기 공개 자료를 이용한 시뮬레이션, 실제 운영에는 일 집계 공급 필요"],
], columns=["항목", "내용"])
display(contract)


,항목,내용
0,예측 대상,서울시 전체의 D일 총 대여건수 (회귀)
1,예측 시점,D-1일 00:00
2,과거 정보,"D-2일까지의 확정 대여건수, D일 달력"
3,활용,도시 전체 운영 규모 계획의 참고 지표
4,데이터 범위,"2025-01-01~2026-06-30, 546일"
5,제약,"반기 공개 자료를 이용한 시뮬레이션, 실제 운영에는 일 집계 공급 필요"


- 기존 분석의 계절·요일 패턴을 일별 수요 예측 문제로 연결함.
- 달력과 과거 대여건수를 사용하며 당일 관측 날씨는 입력에서 제외함.
- 도시 전체 총량 예측으로 개별 대여소 수요를 직접 판단하기 어려움.
- 반기 공개자료의 운영 시뮬레이션이며 실서비스에는 일 집계 공급이 필요함.